In [0]:
import time
from functools import reduce
from operator import or_

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DateType, DoubleType, IntegerType


def _valor(c):
    """trim + string vazia vira NULL (ausencia uniforme na silver)"""
    return F.nullif(F.trim(c), F.lit(""))


def _zero_se_negativo(c, t):
    """cast tipado; valor negativo -> 0 (decisao do professor, 2026-11-09:
    negativo em contagem e artefato do gerador — 'nao era pra ter negativo').
    NULL preservado: condicao NULL cai no otherwise e devolve o cast."""
    cast = F.trim(c).try_cast(t)
    return F.when(cast < 0, F.lit(0)).otherwise(cast)


t0 = time.time()

# ----------------------------------------------------------------
# 1. LEITURA UNICA da bronze
# ----------------------------------------------------------------
bronze = spark.read.table("vidaplus.bronze.fact_recursos_leitos_equipes")

# ----------------------------------------------------------------
# 2. CHECKS DE CONTRATO (EDA 2026-11-06: bronze ja veio limpa)
#    Taxonomia (plano.md E1.8): impossivel -> QUARENTENA;
#    documentado como possivel (ocupados > operacionais, taxa > 1:
#    o dicionario F4a diz que PODE ocorrer) -> ACEITAR;
#    derivavel barato e ja validado (taxa = ocup/oper, 0 divergencias
#    na EDA) -> MANTER do fornecedor;
#    negativo em contagem/medida (artefato do gerador — decisao do
#    professor, 2026-11-09) -> ZERAR (_zero_se_negativo no select)
#    + contador no log; NAO quarentena. Bronze preserva os originais.
# ----------------------------------------------------------------
COLS_TIPO = {
    "leitos_clinicos_operacionais": IntegerType(),
    "leitos_clinicos_ocupados": IntegerType(),
    "taxa_ocupacao_clinica": DoubleType(),
    "leitos_uti_operacionais": IntegerType(),
    "leitos_uti_ocupados": IntegerType(),
    "taxa_ocupacao_uti": DoubleType(),
    "pacientes_aguardando_leito": IntegerType(),
    "tempo_medio_espera_leito_horas": DoubleType(),
    "medicos_escalados": IntegerType(),
    "enfermagem_escalada": IntegerType(),
    "taxa_absenteismo_equipe": DoubleType(),
    "horas_extras_realizadas": IntegerType(),
    "ventiladores_disponiveis": IntegerType(),
    "ano_epidemiologico": IntegerType(),
    "semana_epidemiologica": IntegerType(),
}

# Regra 1: valor presente que nao converte p/ o tipo do contrato
cast_invalido = reduce(
    or_,
    [_valor(c).isNotNull() & _valor(c).try_cast(t).isNull() for c, t in COLS_TIPO.items()],
)

# Regra 2: valor negativo — artefato do gerador (EDA 2026-11-06: 5
# linhas com ventiladores_disponiveis = -1/-2). Decisao do professor
# (2026-11-09): zerar no select; a flag aqui so alimenta o log.
negativo = reduce(or_, [F.trim(c).try_cast(t) < 0 for c, t in COLS_TIPO.items()])

# Regra 0: chave obrigatoria do grao (data valida + unidade presente)
_data_chave = F.trim("data").try_cast(DateType())
_unid_chave = _valor("id_unidade")
chave_ausente = _data_chave.isNull() | _unid_chave.isNull()

# Regra 3: grao duplicado (data x unidade) -> TODAS as linhas do grao quarentenam
w_grao = Window.partitionBy(_data_chave, _unid_chave)

checked = (
    bronze
    .withColumn("_chave_ausente", chave_ausente)
    .withColumn("_cnt_grao", F.count(F.lit(1)).over(w_grao))
    .withColumn("_cast_invalido", cast_invalido)
    .withColumn("_negativo", negativo)
    .cache()
)

violou = F.coalesce(
    F.col("_chave_ausente") | (F.col("_cnt_grao") > 1) | F.col("_cast_invalido"),
    F.lit(False),
)

# ----------------------------------------------------------------
# 3. SILVER: espelho tipado (trim + try_cast); regras semanticas ficam p/ gold
# ----------------------------------------------------------------
silver = checked.filter(~violou).select(
    F.trim("data").try_cast(DateType()).alias("data"),
    F.trim("id_unidade").alias("id_unidade"),  # nao-vazio garantido pela Regra 0
    _valor("cod_municipio_ibge").alias("cod_municipio_ibge"),  # STRING: preserva zeros a esquerda (idem silver F2)
    _valor("uf").alias("uf"),
    _zero_se_negativo("leitos_clinicos_operacionais", IntegerType()).alias("leitos_clinicos_operacionais"),
    _zero_se_negativo("leitos_clinicos_ocupados", IntegerType()).alias("leitos_clinicos_ocupados"),
    _zero_se_negativo("taxa_ocupacao_clinica", DoubleType()).alias("taxa_ocupacao_clinica"),
    _zero_se_negativo("leitos_uti_operacionais", IntegerType()).alias("leitos_uti_operacionais"),
    _zero_se_negativo("leitos_uti_ocupados", IntegerType()).alias("leitos_uti_ocupados"),
    _zero_se_negativo("taxa_ocupacao_uti", DoubleType()).alias("taxa_ocupacao_uti"),
    _zero_se_negativo("pacientes_aguardando_leito", IntegerType()).alias("pacientes_aguardando_leito"),
    _zero_se_negativo("tempo_medio_espera_leito_horas", DoubleType()).alias("tempo_medio_espera_leito_horas"),
    _zero_se_negativo("medicos_escalados", IntegerType()).alias("medicos_escalados"),
    _zero_se_negativo("enfermagem_escalada", IntegerType()).alias("enfermagem_escalada"),
    _zero_se_negativo("taxa_absenteismo_equipe", DoubleType()).alias("taxa_absenteismo_equipe"),
    _zero_se_negativo("horas_extras_realizadas", IntegerType()).alias("horas_extras_realizadas"),
    _zero_se_negativo("ventiladores_disponiveis", IntegerType()).alias("ventiladores_disponiveis"),
    _zero_se_negativo("ano_epidemiologico", IntegerType()).alias("ano_epidemiologico"),
    _zero_se_negativo("semana_epidemiologica", IntegerType()).alias("semana_epidemiologica"),
    F.col("_arquivo_origem"),
    F.col("_ingerido_em"),
)

silver.write.mode("overwrite").saveAsTable("vidaplus.silver.fact_recursos_leitos_equipes")

# ----------------------------------------------------------------
# 4. QUARENTENA: linhas que violaram regra (dado BRUTO, sem transformacao)
#    Precedencia da _regra_violada: chave > grao > cast
#    (negativo saiu da quarentena: zerado no select — regra do professor)
# ----------------------------------------------------------------
quarentena = (
    checked.filter(violou)
    .select(
        F.when(F.col("_chave_ausente"), F.lit("chave_obrigatoria_ausente"))
         .when(F.col("_cnt_grao") > 1, F.lit("grao_duplicado"))
         .when(F.col("_cast_invalido"), F.lit("cast_invalido"))
         .alias("_regra_violada"),
        F.col("_cnt_grao").alias("_total_no_grao"),
        *bronze.columns,
        F.current_timestamp().alias("_data_quarentena"),
    )
)

quarentena.write.mode("overwrite").saveAsTable("vidaplus.silver.quarentena_recursos_leitos_regras_violadas")

# ----------------------------------------------------------------
# 5. LOG da carga
# ----------------------------------------------------------------
n_bronze = bronze.count()
n_silver = silver.count()
n_quarentena = quarentena.count()
n_zerados = checked.filter(F.col("_negativo")).count()

print(f"Bronze:      {n_bronze:>8,} linhas")
print(f"Silver:      {n_silver:>8,} linhas | esperado 21.912 | negativos zerados: {n_zerados} (originais preservados na bronze)")
print(f"Quarentena:  {n_quarentena:>8,} linhas | taxa: {n_quarentena / n_bronze:.2%}")
print(f"Tempo total: {time.time() - t0:.1f}s")

# Invariante de nao-perda: toda linha da bronze esta na silver OU na quarentena
assert n_silver + n_quarentena == n_bronze, (
    f"PERDA DE LINHAS: bronze={n_bronze}, silver+quarentena={n_silver + n_quarentena}"
)

checked.unpersist()  # libera o cache

In [0]:
import time
from functools import reduce
from operator import or_

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DateType, DoubleType, IntegerType


def _valor(c):
    """trim + string vazia vira NULL (ausencia uniforme na silver)"""
    return F.nullif(F.trim(c), F.lit(""))


t0 = time.time()

# ----------------------------------------------------------------
# 1. LEITURA UNICA da bronze
# ----------------------------------------------------------------
bronze = spark.read.table("vidaplus.bronze.fact_recursos_estoque_insumos")

# ----------------------------------------------------------------
# 2. CHECKS DE CONTRATO
#    Taxonomia (plano.md E1.8): impossivel/contraditorio -> QUARENTENA;
#    cobertura <= 0 NAO e violacao (1.813 dias de ruptura com estoque=0
#    + 31 de quase-ruptura com cobertura arredondada a 0) -> ACEITAR;
#    quebra da continuidade estoque_t = estoque_(t-1) + rec - cons
#    (4.838 linhas; artefato do gerador, NAO e contrato do dicionario)
#    -> documentada no plano.md, sem acao na silver.
# ----------------------------------------------------------------
INT_COLS = ["consumo_dia", "quantidade_recebida", "estoque_final", "ponto_de_pedido"]

# Regra 1: valor presente que nao converte p/ o tipo do contrato
cast_invalido = reduce(
    or_,
    [_valor(c).isNotNull() & _valor(c).try_cast(t).isNull()
     for c, t in {**{c: IntegerType() for c in INT_COLS},
                  "dias_de_cobertura": DoubleType(),
                  "ruptura_estoque": IntegerType(),
                  "custo_unitario_brl": DoubleType()}.items()],
)

# Regra 2: fisicamente impossivel — quantidades negativas ou custo nao-positivo
negativo = reduce(
    or_,
    [F.trim(c).try_cast(IntegerType()) < 0 for c in INT_COLS]
    + [F.trim("dias_de_cobertura").try_cast(DoubleType()) < 0],
)
custo_nao_positivo = F.trim("custo_unitario_brl").try_cast(DoubleType()) <= 0
fisica_impossivel = negativo | custo_nao_positivo

# Regra 3: dominio documentado — ruptura 0/1 (dicionario)
ruptura_fora_dominio = ~F.trim("ruptura_estoque").try_cast(IntegerType()).isin(0, 1)

# Regra 4: ruptura incoerente com o estoque (dicionario: ruptura = "saldo
# zerado no dia"). EDA 2026-11-06: 15 linhas com estoque=0 e ruptura=0.
_rupt = F.trim("ruptura_estoque").try_cast(IntegerType())
_est = F.trim("estoque_final").try_cast(IntegerType())
ruptura_incoerente = ((_rupt == 0) & (_est == 0)) | ((_rupt == 1) & (_est > 0))

# Regra 0: chave obrigatoria do grao (data + unidade + insumo)
_data_chave = F.trim("data").try_cast(DateType())
_unid_chave = _valor("id_unidade")
_insumo_chave = _valor("cod_insumo")
chave_ausente = _data_chave.isNull() | _unid_chave.isNull() | _insumo_chave.isNull()

# Regra 5: grao duplicado (data x unidade x insumo) -> TODAS as linhas quarentenam
w_grao = Window.partitionBy(_data_chave, _unid_chave, _insumo_chave)

checked = (
    bronze
    .withColumn("_chave_ausente", chave_ausente)
    .withColumn("_cnt_grao", F.count(F.lit(1)).over(w_grao))
    .withColumn("_cast_invalido", cast_invalido)
    .withColumn("_fisica_impossivel", fisica_impossivel)
    .withColumn("_ruptura_fora_dominio", ruptura_fora_dominio)
    .withColumn("_ruptura_incoerente", ruptura_incoerente)
    .cache()
)

violou = F.coalesce(
    F.col("_chave_ausente") | (F.col("_cnt_grao") > 1) | F.col("_cast_invalido")
    | F.col("_fisica_impossivel") | F.col("_ruptura_fora_dominio") | F.col("_ruptura_incoerente"),
    F.lit(False),
)

# ----------------------------------------------------------------
# 3. SILVER: espelho tipado (trim + try_cast); regras semanticas ficam p/ gold
# ----------------------------------------------------------------
silver = checked.filter(~violou).select(
    F.trim("data").try_cast(DateType()).alias("data"),
    F.trim("id_unidade").alias("id_unidade"),  # nao-vazio garantido pela Regra 0
    F.trim("cod_insumo").alias("cod_insumo"),  # idem
    _valor("nome_insumo").alias("nome_insumo"),
    _valor("unidade_medida").alias("unidade_medida"),
    F.trim("consumo_dia").try_cast(IntegerType()).alias("consumo_dia"),
    F.trim("quantidade_recebida").try_cast(IntegerType()).alias("quantidade_recebida"),
    F.trim("estoque_final").try_cast(IntegerType()).alias("estoque_final"),
    F.trim("ponto_de_pedido").try_cast(IntegerType()).alias("ponto_de_pedido"),
    F.trim("dias_de_cobertura").try_cast(DoubleType()).alias("dias_de_cobertura"),
    F.trim("ruptura_estoque").try_cast(IntegerType()).alias("ruptura_estoque"),
    F.trim("custo_unitario_brl").try_cast(DoubleType()).alias("custo_unitario_brl"),
    F.col("_arquivo_origem"),
    F.col("_ingerido_em"),
)

silver.write.mode("overwrite").saveAsTable("vidaplus.silver.fact_recursos_estoque_insumos")

# ----------------------------------------------------------------
# 4. QUARENTENA: linhas que violaram regra (dado BRUTO, sem transformacao)
#    Precedencia: chave > grao > cast > fisica > dominio > coerencia
# ----------------------------------------------------------------
quarentena = (
    checked.filter(violou)
    .select(
        F.when(F.col("_chave_ausente"), F.lit("chave_obrigatoria_ausente"))
         .when(F.col("_cnt_grao") > 1, F.lit("grao_duplicado"))
         .when(F.col("_cast_invalido"), F.lit("cast_invalido"))
         .when(F.col("_fisica_impossivel"), F.lit("valor_fisicamente_impossivel"))
         .when(F.col("_ruptura_fora_dominio"), F.lit("fora_dominio_documentado"))
         .when(F.col("_ruptura_incoerente"), F.lit("ruptura_incoerente_estoque"))
         .alias("_regra_violada"),
        F.col("_cnt_grao").alias("_total_no_grao"),
        *bronze.columns,
        F.current_timestamp().alias("_data_quarentena"),
    )
)

quarentena.write.mode("overwrite").saveAsTable("vidaplus.silver.quarentena_estoque_insumos_regras_violadas")

# ----------------------------------------------------------------
# 5. LOG da carga
# ----------------------------------------------------------------
n_bronze = bronze.count()
n_silver = silver.count()
n_quarentena = quarentena.count()
n_incoerente = quarentena.filter(F.col("_regra_violada") == "ruptura_incoerente_estoque").count()

print(f"Bronze:      {n_bronze:>8,} linhas")
print(f"Silver:      {n_silver:>8,} linhas | esperado 219.105 (= 219.120 - 15 zeradas sem ruptura)")
print(f"Quarentena:  {n_quarentena:>8,} linhas | ruptura_incoerente: {n_incoerente:,} | taxa: {n_quarentena / n_bronze:.2%}")
print(f"Tempo total: {time.time() - t0:.1f}s")

# Invariante de nao-perda: toda linha da bronze esta na silver OU na quarentena
assert n_silver + n_quarentena == n_bronze, (
    f"PERDA DE LINHAS: bronze={n_bronze}, silver+quarentena={n_silver + n_quarentena}"
)

checked.unpersist()  # libera o cache